# Dataset exploration
Run from the project root after `python -m src.run_all --only data` (or the full pipeline).
Checks label balance, sentence length, vocabulary coverage and code-mixing for both variants.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from src.config import DATA, PROCESSED
from src.pipeline import code_mixing_index
pd.set_option('display.max_colwidth', 120)

## 1. Token-level language labels (`data/language_id.csv`)

In [ ]:
lid = pd.read_csv(DATA / 'language_id.csv', keep_default_na=False)
print(lid.groupby(['variant', 'split']).sentence_id.nunique().unstack())
(lid[lid.split == 'train'].groupby('variant').language.value_counts(normalize=True).unstack().round(3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for a, v in zip(ax, ['hinglish', 'tanglish']):
    lid[(lid.variant == v) & (lid.split == 'train')].language.value_counts().plot.bar(ax=a, title=f'{v}: train token labels')
plt.tight_layout()

## 2. Code-mixing in the training sentences

In [ ]:
cmi = {}
for v in ['hinglish', 'tanglish']:
    d = lid[(lid.variant == v) & (lid.split == 'train') & (lid.kind == 'WORD')]
    cmi[v] = d.groupby('sentence_id').language.apply(lambda s: code_mixing_index(s.tolist()))
pd.DataFrame({v: c.describe() for v, c in cmi.items()}).round(2)

In [ ]:
for v, c in cmi.items():
    print(v, f'{(c > 0).mean():.1%} of sentences contain more than one language')

## 3. Language-model corpora (`data/processed/*.txt`)

In [ ]:
rows = []
for v in ['hinglish', 'tanglish']:
    for s in ['train', 'val', 'test']:
        lines = (PROCESSED / f'{v}_{s}.txt').read_text(encoding='utf-8').splitlines()
        lens = [len(l.split()) for l in lines]
        rows.append(dict(variant=v, split=s, sentences=len(lines), tokens=sum(lens), mean_len=np.mean(lens), types=len({w for l in lines for w in l.split()})))
pd.DataFrame(rows)

In [ ]:
# Vocabulary coverage: what fraction of test tokens appear >= 3 times in train?
for v in ['hinglish', 'tanglish']:
    tr = Counter(w for l in (PROCESSED / f'{v}_train.txt').read_text(encoding='utf-8').splitlines() for w in l.split())
    te = [w for l in (PROCESSED / f'{v}_test.txt').read_text(encoding='utf-8').splitlines() for w in l.split()]
    print(v, 'test tokens covered by train (count>=3):', f"{np.mean([tr[w] >= 3 for w in te]):.1%}")

## 4. Most frequent words per language label (train)

In [ ]:
for v in ['hinglish', 'tanglish']:
    d = lid[(lid.variant == v) & (lid.split == 'train') & (lid.kind == 'WORD')]
    print(v)
    print(d.assign(t=d.token.str.lower()).groupby('language').t.apply(lambda s: ', '.join(s.value_counts().head(15).index)).to_string(), '\n')

## 5. Normalization pairs (`data/normalization.csv`)

In [ ]:
norm = pd.read_csv(DATA / 'normalization.csv', keep_default_na=False)
print(norm.groupby(['variant', 'split']).size())
norm[norm.split == 'test'].sample(5, random_state=1)[['original_text', 'normalized_text']]